In [ ]:
%load_ext autoreload
%autoreload 2

In [ ]:
from general_solve import shape_functions as sf
from fractions import Fraction

In [ ]:
spline_func_map = {0:sf.phi0,1:sf.bspline2,2:sf.bspline3,3:sf.bspline4}
spline_dx_map = {0:sf.phi0_dx,1:sf.bspline2_dx,2:sf.bspline3_dx,3:sf.bspline4_dx}

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

In [ ]:
p_square_xi = lambda nu,rho,i:nu+i
p_trap_xi = lambda nu,rho,i:(2*nu+rho-nu*rho)/(2+rho)+i
p_tri_xi = lambda a,b,i: a/(2+a+b)-b/(2+a)+i
p_square_eta = lambda nu,rho,j:rho+j
p_trap_eta = lambda nu,rho,j: rho/2+j
p_tri_eta = lambda a,b,j:(a+b)/2+j


In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import sys  
import tqdm
from scipy import integrate as scint
from tqdm import tqdm
mypath = '/home/bbb/Galerkin-Differencing/general_solve'
sys.path.insert(1, mypath)
from general_solve.variable import SingleComponentVariable as Var
from general_solve.run_convergence import run_it, plot_it
from general_solve.debug import matvis
from general_solve import globals,debug

In [ ]:
globals.init()

In [ ]:
doflocs = ['node','cell','xside','yside']
rtypes = ['uniform','stripe','square']
rnames = {'uniform':['no'],
		  'stripe':['vertfinecenter',
					'vertcoarsecenter',
					'horzfinecenter',
					'horzcoarsecenter'],
		  'square':['finecenter','coarsecenter']}

u0 = lambda	x: np.sin(2*np.pi*x)+np.cos(2*np.pi*x)
ufunc =	lambda x,y:	u0(x) +	u0(y)
f_lap =	lambda x,y:	-4*np.pi**2*ufunc(x,y)
f_helm = lambda x,y: f_lap(x,y) + ufunc(x,y)

# setup u,v,p,f
mu = 1
rho = 1

u = lambda x,y: -2*np.cos(2*np.pi*x)*np.sin(2*np.pi*y)
ulap = lambda x,y: -8*np.pi**2*u(x,y)

v = lambda x,y: 2*np.sin(2*np.pi*x)*np.cos(2*np.pi*y)
vlap = lambda x,y: -8*np.pi**2*v(x,y)

p = lambda x,y: -np.cos(4*np.pi*x)+np.cos(4*np.pi*y)

f_0 = lambda x,y: 4*np.pi*(2*np.pi*u(x,y)-np.sin(4*np.pi*x))
f_1 = lambda x,y: 4*np.pi*(2*np.pi*v(x,y)+np.sin(4*np.pi*y))

In [ ]:
globals.LAG = False

In [ ]:
globals.DEBUG = True

In [ ]:
a = np.random.random((4,4,2,2))

a[2,3] = np.array([[1,1],[1,1]])

In [ ]:
a

In [ ]:
N = 8
Zvars = []
for v in [1]:
	dofloc = 'xside' if v==0 else 'yside'
	ords = [2,1] if v==0 else [1,2]
	rname = 'vertfinecenter' if v==0 else 'horzfinecenter'
	Zvar = Var(N,dofloc=dofloc,var=ufunc,ords=ords,rtype='stripe',rname=rname,zigzag=True)
	Zvars.append(Zvar)


In [ ]:
tmp = Zvars[0]
tst = np.random.random(len(tmp.constraints.true_dofs))
tst = np.ones_like(tst)
print(tmp.constraints.spC.shape,tst.shape)
full_tst = tmp.constraints.spC.dot(tst)

mysol = tmp.sol(full_tst)

In [ ]:
eps = 1e-2
dom = np.linspace(-eps,eps,1000)
fig,ax = plt.subplots(1,2,figsize=(10,5))
for j in range(2):
	x0 = .25+j/2
	for _ in range(15):
		y = np.random.random(1)[0]
		vals = [mysol([x0+x,y]) for x in dom]
		ax[j].plot(x0+dom,vals)
	ax[j].set_title(x0)
plt.show()

In [ ]:
dom = np.linspace(0,1,1000)
eps = 1e-3
fig,ax = plt.subplots(1,2,figsize=(10,5))
for j in range(2):
	valsL = [mysol([.25+j/2-eps,x]) for x in dom]
	# ax[j].plot(dom,vals,label='LEFT')
	valsR = [mysol([.25+j/2+eps,x]) for x in dom]
	# ax[j].plot(dom,vals,label='RIGHT')
	# ax[j].legend()
	errs = [max(abs(vL-vR)-10*eps,0) for (vL,vR) in zip(valsL,valsR)]
	# scaled_errs = []
	# for err in errs:
	# 	if err < eps:
	# 		scaled_errs.append(0)
	# 	else:
	# 		scaled_errs.append()
	ax[j].plot(dom,errs)
	ax[j].set_title(.25+j/2)
plt.show()

In [ ]:
tmp._setup_mean_value()

In [ ]:
tmp = Zvars[0]
# tmp.mesh.view_detailed(split=False)

In [ ]:
all_els = list(tmp.mesh.patches[0].elements.values())#+list(tmp.mesh.patches[1].elements.values())
el_ids = []

for e in all_els:
	if e.ID in el_ids:
		print(e.ID)
	else:
		el_ids.append(e.ID)

In [ ]:
dom = np.linspace(1e-13,1-1e-13)
for dof in tmp.mesh.patches[0].dofs.values():
	if dof.interface:# and dof.x==.25:
		xs,ys,vs,v2s = [],[],[],[]
		# plt.plot(dof.x,dof.y,'ko')
		# ecnt = 0
		for e in dof.elements.values():
			# print(e.global_ID)
			# plt.plot(e.x,e.y,'.')
			for a in dom:
				for b in dom:
					if e.regular or not e.tri or a+b<1:
						x,y = e.transform(a,b)

						checks = []
						for el in dof.elements.values():
							if el.check_loc([x,y]): checks.append(el.ID)
						if len(checks) == 1:
							assert checks[0] == e.ID
						else:
							print(e.ID,checks,sep='\t')
						# plt.plot(x,y,'o')
						# try:
						# 	assert e.check_loc([x,y])
						# except:
						# 	print(e.regular)
						v = dof.phi([x,y],el=e,glob=True)
						v2 = dof.phi([x,y],glob=True)
						xs.append(x)
						ys.append(y)
						vs.append(v)
						v2s.append(v2)
						# plt.plot(x,y,'C{}o'.format(ecnt%10))
						# if not e.regular:
						# 	xi,eta = e.phi_input_global(x,y)
						# 	plt.plot(xi,eta,'o')
			# plt.plot(e.to_plot[0],e.to_plot[1],'w')
			# if not e.regular:
			# 	plt.show()
			# ecnt += 1
		# plt.show()
		plt.subplot(121)
		plt.scatter(xs,ys,c=vs)
		plt.colorbar()
		for e in dof.elements.values():
			plt.plot(e.to_plot[0],e.to_plot[1],'w')
		plt.subplot(122)
		plt.scatter(xs,ys,c=v2s)
		plt.colorbar()
		for e in dof.elements.values():
			plt.plot(e.to_plot[0],e.to_plot[1],'w')
		plt.show()
		# break

In [ ]:
e = tmp.mesh.zigzag_elements[3]
start = [e.x,e.y]
step = e.h

xdom = np.linspace(start[0]-2*step,start[0]+3*step)
ydom = np.linspace(start[1]-2*step,start[1]+3*step)

fig,ax = plt.subplots(1,2,figsize=(10,5))
for x in xdom:
	for y in ydom:
		axid = int(e.check_loc([x,y]))
		ax[axid].plot(x,y,'o')
# plt.plot(e.to_)
# for xi in dom:
# 	if xi >= .5:
# 		for eta in dom:
# 			x,y = e.gtransform(xi,eta)
# 			plt.plot(x,y,'o')
for _ in range(2):
	ax[_].plot(e.to_plot[0],e.to_plot[1],'k')
plt.show()

# for nu in dom:
# 	for rho in dom:
# 		x,y = e.transform(nu,rho)
# 		xi,eta = e.ginv_transform(x,y)
# 		plt.plot(xi,eta,'o')
# # plt.plot(e.to_plot[0],e.to_plot[1])
# plt.show()


In [ ]:
for e in tmp.mesh.zigzag_elements:
	if e.tri:
		print(e.trap_count)
		plt.plot(e.to_plot[0],e.to_plot[1])
		for dof in e.dof_list:
			print(dof.x,dof.y)
			plt.plot(dof.x,dof.y,'o')
		plt.show()

In [ ]:
e = tmp.mesh.zigzag_elements[-4]

In [ ]:
plt.plot(e.to_plot[0],e.to_plot[1])
for dof in e.dof_list:
	print(dof.x,dof.y)
	plt.plot(dof.x,dof.y,'o')
plt.show()

In [ ]:
tris,traps = [],[]
for Zvar in Zvars:
	tris.append(Zvar.mesh.zigzag_elements[-15])
	tris.append(Zvar.mesh.zigzag_elements[-2])
	traps.append(Zvar.mesh.zigzag_elements[2])
	traps.append(Zvar.mesh.zigzag_elements[15])

In [ ]:
dom = np.linspace(0,1,15)
sc = 3*4*5
dha = {0.:'right',1.:'left'}
dva = {0.:'top',1.:'bottom'}
cstar = {0:(0,0),
		 2:(1,0),
		 1:(0,0),
		 3:(0,1)}
for tri in tris+traps:
	print(tri.map_type)
	fig,ax = plt.subplots(1,4,figsize=(40,10))

	if len(tri.corners) == 3:
		c0,c1 = cstar[tri.map_type]
		X = (tri.K+c0)*tri.h
		Y = (tri.L+c1)*tri.h
		ax[0].plot(X,Y,'ko',ms=20)

	xs,ys = [],[]
	gxs,gys = [],[]
	for xi in dom:
		for eta in dom:
			noskip = len(tri.corners) == 4
			if noskip or (xi+eta)<=1:
				x,y = tri.transform(xi,eta)
				xs.append(x)
				ys.append(y)
				ax[0].plot(x,y,'.')

				xin,yin = tri.phi_input_global(x,y)
				ax[2].plot(xin,yin,'.')

				xin,yin = tri.phi_input_local(xi,eta)
				ax[3].plot(xin,yin,'.')
				# if noskip:
				# 	gx,gy = tri.gtransform(xi,eta)
				# 	gxs.append(gx)
				# 	gys.append(gy)
				# 	ax[1].plot(gx,gy,'.')
				# else:
				# 	gxs.append(x)
				# 	gys.append(y)
				# nu,eta = tri.phi_input_local(xi,eta)
				# ax[2].plot(nu,eta,'.')
	for j in [0]:
		ax[j].plot(tri.to_plot[0],tri.to_plot[1])


	for (x,y) in zip(xs,ys):
		xin,yin = tri.inv_transform(x,y)#phi_input_global(x,y)
		ax[1].plot(xin,yin,'.')
	for (gx,gy) in zip(gxs,gys):
		gxin,gyin = tri.ginv_transform(gx,gy)#phi_input_global(x,y)
		ax[3].plot(gxin,gyin,'.')
	for j in range(3):
		ax[j+1].plot([0,0,1,1,0],[0,1,1,0,0],'k')
	# ax[2].plot([0,0,1,1,0],[0,1,1,0,0],'k')
	# for j in [1,2]:
	# 	ax[j].plot([.5,.5,.25,.75],[.25,.75,.5,.5],'k.')

	for j,c in enumerate(tri.corners):
		test0,test1 = tri.inv_transform(c.x,c.y)#phi_input_global(c.x,c.y)
		ax[0].plot(c.x,c.y,'C'+str(j)+'o',markeredgecolor='k',ms=10)
		ax[1].plot(test0,test1,'C'+str(j)+'o',markeredgecolor='k',ms=10)
		phitests = [tri.phi_input_global(c.x,c.y),tri.phi_input_local(test0,test1)]
		for i,(gtest0,gtest1) in enumerate(phitests):
			ax[i+2].plot(gtest0,gtest1,'C'+str(j)+'o',markeredgecolor='k',ms=10)
			frac0,frac1 = int(gtest0*sc),int(gtest1*sc)
			try:
				assert (frac0==gtest0*sc) and (frac1==gtest1*sc)
			except:
				print(gtest0,gtest1)
			ha = 'left' if gtest0>0 else 'right'
			ha = ha if gtest0 not in dha else dha[gtest0]	
			va = 'bottom' if gtest1>0 else 'top'
			va = va if gtest1 not in dva else dva[gtest1]	

			ax[i+2].annotate('({},{})'.format(Fraction(frac0,sc),Fraction(frac1,sc)),
				 					(gtest0,gtest1),(gtest0,gtest1),fontsize=15,ha=ha,va=va)

	for j in [0,1,2,3]:
		ax[j].axis('off')
		ax[j].set_aspect('equal')
	plt.show()

In [ ]:
dom = np.linspace(0,1,15)
sc = 3*4*5
dha = {0.:'right',1.:'left'}
dva = {0.:'top',1.:'bottom'}
for tri in tris:#tris+traps:
	print(tri.map_type)
	fig,ax = plt.subplots(1,4,figsize=(40,10))

	xs,ys = [],[]
	gxs,gys = [],[]
	for alpha in dom:
		for beta in dom:
			if len(tri.corners) == 4 or (alpha+beta)<=1:
				x,y = tri.transform(alpha,beta)
				# gx,gy = tri.gtransform(xi,eta)
				xs.append(x)
				ys.append(y)
				# gxs.append(gx)
				# gys.append(gy)
				if alpha == 0:
					col = 'C3o'
				elif beta==0:
					col = 'C4o'
				else:
					col='k.'
				ax[0].plot(x,y,col)#'.')
				# ax[1].plot(gx,gy,'.')
				# nu,eta = tri.phi_input_local(xi,eta)
				# ax[2].plot(nu,eta,'.')
	for j in [0,1]:
		ax[j].plot(tri.to_plot[0],tri.to_plot[1])

	for j,c in enumerate(tri.corners):
		for i in range(2):
			ax[i].plot(c.x,c.y,'C'+str(j)+'o',markeredgecolor='k',ms=10)

	# for (x,y) in zip(xs,ys):
	# 	xin,yin = tri.inv_transform(x,y)#phi_input_global(x,y)
	# 	ax[2].plot(xin,yin,'.')
	# for (gx,gy) in zip(gxs,gys):
	# 	gxin,gyin = tri.ginv_transform(gx,gy)#phi_input_global(x,y)
	# 	ax[3].plot(gxin,gyin,'.')
	# for j in range(2):
	# 	ax[j+2].plot([0,0,1,1,0],[0,1,1,0,0],'k')
	# # ax[2].plot([0,0,1,1,0],[0,1,1,0,0],'k')
	# # for j in [1,2]:
	# # 	ax[j].plot([.5,.5,.25,.75],[.25,.75,.5,.5],'k.')

	# for j,c in enumerate(tri.corners):
	# 	test0,test1 = tri.inv_transform(c.x,c.y)#phi_input_global(c.x,c.y)
	# 	ax[2].plot(test0,test1,'C'+str(j)+'o',markeredgecolor='k',ms=10)
	# 	gtest0,gtest1 = tri.ginv_transform(c.x,c.y)
	# 	ax[3].plot(gtest0,gtest1,'C'+str(j)+'o',markeredgecolor='k',ms=10)
	# 	frac0,frac1 = int(gtest0*sc),int(gtest1*sc)
	# 	assert (frac0==gtest0*sc) and (frac1==gtest1*sc)
	# 	ha = 'center' if gtest0 not in dha else dha[gtest0]	
	# 	va = 'center' if gtest1 not in dva else dva[gtest1]	
	# 	ax[3].annotate('({},{})'.format(Fraction(frac0,sc),Fraction(frac1,sc)),
	# 			 					(gtest0,gtest1),(gtest0,gtest1),fontsize=15,ha=ha,va=va)
	for j in [0,1,2,3]:
		ax[j].axis('off')
		ax[j].set_aspect('equal')
	plt.show()

In [ ]:
fig,ax = plt.subplots(1,2,figsize=(20,10))
dom = np.linspace(0,1)
for xi in dom:
	dom2 = np.linspace(0,1-xi)
	for eta in dom2:
		x,y = tri.transform(xi,eta)
		ax[0].plot(xi,eta,'k.')
		ax[1].plot(x,y,'k.')

j = 0
for corner in [(0,0),(0,1),(1,0)]:
	xi,eta = corner
	x,y = tri.transform(xi,eta)
	ax[0].plot(xi,eta,'C{}o'.format(j),ms=15)
	ax[1].plot(x,y,'C{}o'.format(j),ms=15)
	j += 1

for side in [(0,.5),(.5,0),(.5,.5)]:
	xi,eta = side
	x,y = tri.transform(xi,eta)
	ax[0].plot(xi,eta,'C{}o'.format(j),ms=15)
	ax[1].plot(x,y,'C{}o'.format(j),ms=15)
	j += 1

In [ ]:
var.solve_poisson(f_lap)
Zvar.solve_poisson(f_lap)

In [ ]:
v = Zvar

dom = np.linspace(0,1,201)
locs = np.array([a.flatten() for a in np.meshgrid(dom,dom)]).T
my_coefs = np.ones_like(v.curr_coefs)
var.vis_dof_sol(var.curr_coefs,locs=locs,err=False,log=False,lines=True)
v.vis_dof_sol(v.curr_coefs,locs=locs,err=False,log=False,lines=True)
v.vis_dof_sol(v.curr_coefs,locs=locs,err=True,log=False,lines=True)
res = v.vis_dof_sol(my_coefs,locs=locs,err=False,log=False,lines=True,ave_only=True)
try:
	assert(abs(res[0]-res[-1])<1e-10)
except:
	print(res)

##### lots of continuity checks

In [ ]:
for v in myvars:
	debug.check_continuity(v,full=True,reps=7)

In [ ]:
tol = 1e-14
err_tol = 10
old_dom = np.linspace(tol,1-tol)

def fwrap(func,comp):
	return lambda loc: func([loc[1-comp],loc[comp]])

def fsupport(func):
	def supported(loc):
		if (tol<loc[0]<1-tol) and (tol<loc[1]<1-tol):
			return func(loc)
		return 0
	return supported

def convert_dom(val,comp,transform):
	coords = []
	for pt in old_dom:
		pair = [val,pt]
		x,y = transform(pair[1-comp],pair[comp])
		coords.append([x,y])
	return coords

def check_reg_el_edges(v,fsol=None):
	if fsol is None:
		randx = np.random.random(len(v.constraints.true_dofs))
		coefs = v.constraints.spC.dot(randx)
		fsol = v.sol(coefs)
	starts = {}
	pairs = []
	problem_spots = {0:[],1:[],2:[],3:[]}
	for p in v.mesh.patches:
		for e_id in p.elements:
			e = p.elements[e_id]
			el_added = False
			corners = [[e.x,e.y],[e.x+e.h,e.y],[e.x,e.y+e.h],[e.x+e.h,e.y+e.h]]
			for c in corners:
				check = [int(c[0]*4/v.h),int(c[1]*4/v.h)]
				if check[0] in starts and check[1] in starts[check[0]]:
					pass
				else:
					pairs.append([[c[0],c[0]],[c[1],c[1]+e.h]])
					pairs.append([[c[0],c[0]+e.h],[c[1],c[1]]])
					for comp in range(2):
						c[1-comp] = min(1-tol,max(tol,c[1-comp]))
						dom = np.linspace(max(tol,c[comp]),min(1-tol,c[comp]+e.h))
						vals = {}
						for ep_sgn in [-1,1]:
							tmp_func = fwrap(fsol,comp)
							vals[ep_sgn] = [tmp_func([c[1-comp]+ep_sgn*tol,b]) for b in dom]
						diffs = [round(abs(v0-v1),err_tol) for (v0,v1) in zip(vals[-1],vals[1])]
						for j,diff in enumerate(diffs):
							if diff > 0:
								problem_spots[comp].append(dom[j])
								problem_spots[1-comp].append(c[1-comp])
								problem_spots[2].append([vals[-1][j],vals[1][j]])
								if not el_added:
									problem_spots[3].append(e)
									el_added = True
					
					if check[0] in starts:
						starts[check[0]].append(check[1])
					else:
						starts[check[0]] = []

	return problem_spots,starts,pairs

def check_irreg_el_edges(v,fsol_in=None):
	if fsol_in is None:
		randx = np.random.random(len(v.constraints.true_dofs))
		coefs = v.constraints.spC.dot(randx)
		fsol_in = v.sol(coefs)
	fsol = fsupport(fsol_in)
	problem_spots = {0:[],1:[],2:[],3:[]}
	for e in v.mesh.zigzag_elements:
		el_added = False
		for comp in range(2):
			for side in [0,1]:
				vals = {}
				for ep_sgn in [-1,1]:
					locs = convert_dom(side+ep_sgn*tol,comp,e.transform)
					vals[ep_sgn] = [fsol(loc) for loc in locs]
				diffs = [round(abs(v0-v1),err_tol) for (v0,v1) in zip(vals[-1],vals[1])]
				for j,diff in enumerate(diffs):
					if diff > 0:
						problem_spots[0].append(locs[j][0])
						problem_spots[1].append(locs[j][1])
						problem_spots[2].append([vals[-1][j],vals[1][j]])
						if not el_added:
							problem_spots[3].append(e)
							el_added = True
	return problem_spots

In [ ]:
my_e=Zvar.mesh.loc_to_el([.8,.95])[0]

In [ ]:
j = np.linspace(0,1,len(Zvar.mesh.all_elements))
# e_ids = [e.global_ID for e in Zvar.mesh.all_elements]
e_ids = {}
for e in Zvar.mesh.all_elements:
	if e.global_ID in e_ids:
		plt.plot(e.to_plot[0],e.to_plot[1])
		other_e = e_ids[e.global_ID]
		plt.plot(other_e.to_plot[0],other_e.to_plot[1],'grey',lw=1)
		plt.plot([e.mid[0],other_e.mid[0]],[e.mid[1],other_e.mid[1]],'k')
	else:
		e_ids[e.global_ID] = e
	# e_ids.append(e.global_ID)
plt.show()
print(len(e_ids),len(list(set(e_ids))))

In [ ]:
reg_probs,starts,pairs = check_reg_el_edges(Zvar)
irreg_probs = check_irreg_el_edges(Zvar)

if len(reg_probs[0]) > 0 or len(irreg_probs[0]) > 0:
	fig = plt.figure(figsize=(10,10))
	plt.plot(my_e.to_plot[0],my_e.to_plot[1],'C1',lw=3)
	for p in pairs:
		plt.plot(p[0],p[1],'k',lw=5,alpha=.2)
	plt.plot([0,.25,.25,.25,.75,.75,.75,1,1,0,0],[0,0,1,0,0,1,0,0,1,1,0],'k')
	for e in Zvar.mesh.all_elements:
		plt.plot(e.to_plot[0],e.to_plot[1],'k',lw=.5)
	for xval in starts:
		yvals = [y/4*Zvar.h for y in starts[xval]]
		plt.plot([xval/4*Zvar.h]*len(yvals),yvals,'C2.')
	plt.plot(reg_probs[0],reg_probs[1],'C0o',ms=1)
	plt.plot(irreg_probs[0],irreg_probs[1],'r.',ms=1)
	plt.show()

In [ ]:
randx = np.random.random(len(Zvar.constraints.true_dofs))
coefs = Zvar.constraints.spC.dot(randx)
fsol = Zvar.sol(coefs)
yvals = np.linspace(my_e.y-my_e.h/2,my_e.y+3*my_e.h/2,1111)
vals0 = [fsol([my_e.x-1e-12,y]) for y in yvals]
vals1 = [fsol([my_e.x+1e-12,y]) for y in yvals]
plt.plot(yvals,vals0,'.',ms=1)
plt.plot(yvals,vals1)

In [ ]:
xs,ys = [reg_probs[i]+irreg_probs[i] for i in [0,1]]
miny,maxy = min(ys),max(ys)
ys = np.linspace(miny-2*Zvar.h,maxy+Zvar.h*2,1111)
xs = np.ones_like(ys)*xs[0]
tol = 1e-14
checked = []
plotted = []
online = []
fig = plt.figure(figsize=(15,6))
count = 0
count_fine = 0
for e in irreg_probs[3]:
	for i,dof in enumerate(e.dof_list):
		if dof not in checked:
			checked.append(dof)
			vals = []
			for ep_sgn in [-1,1]:
				eps = ep_sgn*tol
				vals.append([dof.phi([x+eps,y],glob=True) for (x,y) in zip(xs,ys)])
			diffs = [v1-v0 for (v0,v1) in zip(vals[0],vals[1])]
			diff = max([round(abs(d),12) for d in diffs])
			if diff > 0:
				mmax = max(max(vals[0]),max(vals[1]))
				plotted.append(dof)
				plt.subplot(2,6,count+1)

				plt.plot([miny,miny],[0,mmax],'k:',lw=1)
				plt.plot([maxy,maxy],[0,mmax],'k:',lw=1)
				plt.plot(ys,vals[0],'.',ms=2)
				plt.plot(ys,vals[1],'.',ms=1,alpha=.1)
				plt.title([(dof.x-xs[0])/Zvar.h,round((dof.y-miny)/Zvar.h,5)])
				# plt.plot(ys,diffs,'.',ms=1)
				plt.xticks([miny,maxy],['',''])
				count += 1

			elif dof.x == .75:
				online.append(dof)
				plt.subplot(2,6,count_fine+8)
				plt.plot(ys,vals[0])
				plt.plot(ys,vals[1],':')
				plt.title([(dof.x-xs[0])/Zvar.h,round((dof.y-miny)/Zvar.h,5)])
				# plt.plot(ys,diffs,'.',ms=1)
				plt.xticks([miny,maxy],['',''])
				count_fine += 1

# plt.legend()
plt.show()

for e in irreg_probs[3]:
	plt.plot(e.to_plot[0],e.to_plot[1],'k',lw=1)
for dof in checked:
	plt.plot(dof.x,dof.y,'k.')
for dof in online:
	plt.plot(dof.x,dof.y,'ko')
for dof in plotted:
	plt.plot(dof.x,dof.y,'o')
plt.plot([xs[0],xs[0]],[miny,maxy],'r')
plt.show()

jdom = np.linspace(miny-2*Zvar.h,maxy+2*Zvar.h,1111)
J = len(plotted)
fig = plt.figure(figsize=(J*3,4))
for j,dof in enumerate(plotted):
	plt.subplot(1,J,j+1)
	vals = [dof.phi([.75-1e-14,y]) for y in jdom]
	plt.plot(jdom,vals,'C'+str(j))
	plt.xticks([miny,maxy],['',''])
plt.show()



In [ ]:
xs,ys = [reg_probs[i]+irreg_probs[i] for i in [0,1]]
miny,maxy = min(ys)-3*Zvar.h,max(ys)-3*Zvar.h
ys = np.linspace(miny-2*Zvar.h,maxy+Zvar.h*2,1111)
xs = np.ones_like(ys)*xs[0]
tol = 1e-14
checked = []
plotted = []
online = []
fig = plt.figure(figsize=(15,6))
count = 0
count_fine = 0
eshft = len(Zvar.mesh.patches[1].elements)
new_es = []
for e in irreg_probs[3]:
	tmp = Zvar.mesh.zigzag_elements[e.ID-eshft]
	print(tmp.ID,e.ID)
	new_es.append(Zvar.mesh.zigzag_elements[e.ID-eshft-3])
for e in new_es:
	for i,dof in enumerate(e.dof_list):
		if dof not in checked:
			checked.append(dof)
			vals = []
			for ep_sgn in [-1,1]:
				eps = ep_sgn*tol
				vals.append([dof.phi([x+eps,y],glob=True) for (x,y) in zip(xs,ys)])
			diffs = [v1-v0 for (v0,v1) in zip(vals[0],vals[1])]
			diff = max([round(abs(d),12) for d in diffs])
			if diff > 0:
				mmax = max(max(vals[0]),max(vals[1]))
				plotted.append(dof)
				plt.subplot(2,6,count+1)

				plt.plot([miny,miny],[0,mmax],'k:',lw=1)
				plt.plot([maxy,maxy],[0,mmax],'k:',lw=1)
				plt.plot(ys,vals[0],'.',ms=2)
				plt.plot(ys,vals[1],'.',ms=1,alpha=.1)
				plt.title([(dof.x-xs[0])/Zvar.h,round((dof.y-miny)/Zvar.h,5)])
				# plt.plot(ys,diffs,'.',ms=1)
				plt.xticks([miny,maxy],['',''])
				count += 1

			elif dof.x == .75:
				online.append(dof)
				plt.subplot(2,6,count_fine+8)
				plt.plot(ys,vals[0])
				plt.plot(ys,vals[1],':')
				plt.title([(dof.x-xs[0])/Zvar.h,round((dof.y-miny)/Zvar.h,5)])
				# plt.plot(ys,diffs,'.',ms=1)
				plt.xticks([miny,maxy],['',''])
				count_fine += 1

# plt.legend()
plt.show()

for e in irreg_probs[3]:
	plt.plot(e.to_plot[0],e.to_plot[1],'k',lw=1)
for dof in checked:
	plt.plot(dof.x,dof.y,'k.')
for dof in online:
	plt.plot(dof.x,dof.y,'ko')
for dof in plotted:
	plt.plot(dof.x,dof.y,'o')
plt.plot([xs[0],xs[0]],[miny,maxy],'r')
plt.show()

jdom = np.linspace(miny-2*Zvar.h,maxy+2*Zvar.h,1111)
J = len(plotted)
fig = plt.figure(figsize=(J*3,4))
for j,dof in enumerate(plotted):
	plt.subplot(1,J,j+1)
	vals = [dof.phi([.75-1e-14,y]) for y in jdom]
	plt.plot(jdom,vals,'C'+str(j))
	plt.xticks([miny,maxy],['',''])
plt.show()



In [ ]:
randx = np.random.random(len(Zvar.constraints.true_dofs))
coefs = Zvar.constraints.spC.dot(randx)
myfunc = Zvar.sol(coefs)#Zvar.curr_coefs)
eps = 1e-14
all_xs,all_ys,bad_xs,bad_ys = [],[],[],[]
xshifts = [-Zvar.h,-Zvar.h/2,0,Zvar.h/2,Zvar.h,3*Zvar.h/2,2*Zvar.h]
xlabs = ['-h','-h/2','0','h/2','h','3h/2','2h']
dom = np.linspace(0,1)[1:-1]
tmp = np.linspace(0,1)
eta0 = [[xi,0] for xi in dom]
eta1 = [[xi,1] for xi in dom]
xi0 = [[0,eta] for eta in dom]
xi1 = [[1,eta] for eta in dom]
ops  = [(1,eta0),(1,eta1),(0,xi0),(0,xi1)]
labs = [r'$\eta = 0$',r'$\eta = 1$',r'$\xi = 0$',r'$\xi = 1$']
for e in Zvar.mesh.zigzag_elements:
	cindex = 0
	if not e.tri:
		diffs = []
		for i,(comp,op) in enumerate(ops):
			vals0,vals1 = [],[]
			mydom = []
			diff = 0
			tmpx,tmpy = [],[]
			for loc in op:
				loc0 = loc.copy()
				loc0[comp] -= eps
				loc1 = loc.copy()
				loc1[comp] += eps
				x0,y0 = e.transform(loc0[0],loc0[1])
				x1,y1 = e.transform(loc1[0],loc1[1])
				all_pts = [x0,x1,y0,y1]
				all_good = True
				for pt in all_pts:
					if pt < 0 or pt > 1:
						all_good = False

				if all_good:
					# tmpx.append(x0)
					# tmpy.append(y0)
					mydom.append(loc[1-comp])
					vals0.append(myfunc([x0,y0]))
					vals1.append(myfunc([x1,y1]))
					this_diff = round(abs(vals0[-1]-vals1[-1]),10)
					if this_diff > 0:
						bad_xs.append(x0)
						bad_ys.append(y0)
					else:
						all_xs.append(x0)
						all_ys.append(y0)
					diff = max(diff,this_diff)
			if diff > 0:
				# bad_xs += tmpx
				# bad_ys += tmpy
				diffs.append(diff)
				c = 'C'+str(cindex)
				cindex += 1
				plt.plot(mydom,vals0,c,label=labs[i])
				plt.plot(mydom,vals1,c,lw=10,alpha=.2)
			# else:
				# all_xs += tmpx
				# all_ys += tmpy
		for j,xshft in enumerate(xshifts):
			ymin = max(0,min(e.corners[0].y,e.corners[1].y))
			ydom = np.linspace(ymin,min(1,ymin+2*e.h))
			vals0 = [myfunc([e.x+xshft-eps,y]) for y in ydom]
			vals1 = [myfunc([e.x+xshft+eps,y]) for y in ydom]
			mydiff = [round(abs(v0-v1),10) for (v0,v1) in zip(vals0,vals1)]
			bad_inds = [i for i in range(len(mydiff)) if mydiff[i]>0]
			good_inds = [i for i in range(len(mydiff)) if i not in bad_inds]
			if max(mydiff) > 0:
				bad_xs += [e.x+xshft]*len(bad_inds)
				bad_ys += list(ydom[bad_inds])
				diffs.append(max(mydiff))
				c = ':C'+str(cindex)
				cindex += 1
				plt.plot(tmp,vals0,c,label='x = x0 + '+xlabs[j])
				plt.plot(tmp,vals1,c,lw=10,alpha=.2)
			all_ys += list(ydom[good_inds])
			all_xs += [e.x+xshft]*len(good_inds)
		if len(diffs)>0:
			plt.title(diffs)
			plt.legend()
			plt.show()
			
# full_dom = np.linspace(0,1,1001)[1:-1]
# for x in tqdm(full_dom):
# 	for y in full_dom:
# 		v0 = myfunc([x-eps,y-eps])
# 		v1 = myfunc([x-eps,y+eps])
# 		v2 = myfunc([x+eps,y-eps])
# 		v3 = myfunc([x+eps,y+eps])
# 		diffss = [abs(v0-v1),abs(v0-v2),abs(v0-v3),abs(v1-v2),abs(v1-v3),abs(v2-v3)]
# 		if round(max(diffss),10)>0:
# 			bad_xs.append(x)
# 			bad_ys.append(y)

for e in Zvar.mesh.zigzag_elements:
	if not e.tri:
		plt.plot(e.to_plot[0],e.to_plot[1],'k')
plt.plot([0,.25,.25,.25,.75,.75,.75,1,1,0,0],[0,0,1,0,0,1,0,0,1,1,0],'k')
plt.plot(all_xs,all_ys,'.',ms=1)
plt.plot(bad_xs,bad_ys,'r.',ms=2)
plt.show()


# 	if e.x < .301 < e.x+e.h:
# 		if e.y - e.h < .037 < e.y+2*e.h:
# 			plt.plot(e.to_plot[0],e.to_plot[1])
# 			print(e.check_loc([.301,.037]))
# plt.plot(.301,.037,'o')
# plt.show()


In [ ]:
# shfts = [-Zvar.h,-Zvar.h/2,0,Zvar.h/2,Zvar.h,3*Zvar.h/2,2*Zvar.h]
# shfts = [-1,-,-.75,-.5,-.25,0,.25,.5,.75,1]
vs = list(range(-8,9))#[-8,-7,-6,-4,-2,0,2,4,6,7,8]
shfts = [v/8. for v in vs]
labs = ['{}h/8'.format(v) for v in vs]
for p in [Zvar.mesh.patches[0]]:
	for dof_id in p.dofs:
		if p.dofs[dof_id].interface:
			count_dict = {i:[] for i in range(len(vs))}
			dof = p.dofs[dof_id]
			if dof.x > .5:
				
				xdom = np.linspace(dof.x-dof.h,dof.x+2*dof.h)
				ydom = np.linspace(dof.y-2*dof.h,dof.y+2*dof.h)
				y_ops = [dof.y+shft*Zvar.h for shft in shfts]

				fig = plt.figure(figsize=(10,4))
				plt.subplot(121)
				vals = []
				xs,ys = [],[]
				for x in xdom:
					for y in ydom:
						xs.append(x)
						ys.append(y)
						val = dof.phi([x,y],glob=True)
						vals.append(val)
				
				plt.xlabel('x')
				plt.ylabel('y')
				plt.scatter(xs,ys,c=vals,cmap='jet')
				plt.colorbar()

				for e_id in dof.elements:
					e = dof.elements[e_id]
					plt.plot(e.to_plot[0],e.to_plot[1],'white',lw=.5)
				plt.plot(dof.x,dof.y,'o',c='white')

				count = 0
				for y_op in y_ops:
					if (0<=y_op <=1) and (dof.y-3*dof.h/2 <= y_op <= dof.y+3*dof.h/2):
						plt.plot([xdom[0],xdom[-1]],[y_op,y_op],'C'+str(count%10),lw=3)
						count += 1
				
				plt.subplot(122)
				cont,round = True,0
				old_y,old_h = dof.y,dof.h
				while cont:
					all_good = True
					shft = -Zvar.h/4 if round == 1 else Zvar.h/4
					if round > 0:
						dof_id = Zvar.mesh.patches[1]._get_lookup_id_from_loc([dof.x,old_y+shft])
						if -old_h/2 < old_y+shft < 1+old_h/2:
							dof = Zvar.mesh.patches[1].dofs[dof_id]
						else:
							all_good = False
					if all_good:
						if p.level == 1:
							xdom = np.linspace(max(.25,dof.x-dof.h),min(.75,dof.x+2*dof.h))
						elif round > 0:
							xdom = np.linspace(max(.25+dof.h,dof.x-dof.h),min(.75-dof.h,dof.x+2*dof.h))
						else:
							if dof.x < .5:
								plt.plot([.25+dof.h/2,.25+dof.h/2],[0,1],'k:')
								xdom = np.linspace(dof.x-dof.h,min(.25-1e-15+dof.h/2,dof.x+2*dof.h))
							else:
								plt.plot([.75-dof.h/2,.75-dof.h/2],[0,1],'k:')
						# x_ops = [dof.x+shft for shft in shfts]
						y_ops = [old_y+yshft*Zvar.h for yshft in shfts]

						count = 0
						for y_op,lab in zip(y_ops,labs):
							if (0<=y_op <=1) and (old_y-3*old_h/2 <= y_op <= old_y+3*old_h/2):
								count_dict[count].append(y_op)
								vals = [dof.phi([x,y_op],glob=True)*(1+.005*count) for x in xdom]
								if round == 0:
									plt.plot(xdom,vals,'C'+str(count%10),label=lab)
								elif round == 1:
									plt.plot(xdom,vals,'C'+str(count%10),lw=2)#ls='--')
								else:
									plt.plot(xdom,vals,'C'+str(count%10),ls=':',lw=3)
								count += 1
					if p.level == 1: cont = False#or old_y < 0: cont = False
					if round == 2: cont = False
					round += 1

				for k in count_dict:
					if len(count_dict[k])>0:
						assert min(count_dict[k])==max(count_dict[k])
						# print(k,count_dict[k],sep='\t')
				# plt.legend()
				plt.xlabel('x')
				# plt.xlim([.25+3*Zvar.h/8,.25+Zvar.h])
				# plt.ylim(-.02,.6)

				plt.show()

#### checking element transforms and dof assignments on interface

In [ ]:
for e in Zvar.mesh.all_elements:#zigzag_elements:
	runit = False
	for dof in e.dof_list:
		check = e in dof.elements.values()
		if not check:
			runit = True

	if runit:
		plt.plot(e.to_plot[0],e.to_plot[1],'k')
		for dof in e.dof_list:
			check = e in dof.elements.values()
			m = 'o' if check else '*'
			plt.plot(dof.x,dof.y,m)
		plt.show()

In [ ]:
cmap = {4:'C0',6:'C1',8:'C2'}
for p in Zvar.mesh.patches:
	for dof_id in p.dofs:
		dof = p.dofs[dof_id]
		if dof.interface:
			for e_id in dof.elements:
				e = dof.elements[e_id]
				plt.plot(e.to_plot[0],e.to_plot[1])
			plt.plot(dof.x,dof.y,'k.')
			plt.title(len(dof.elements))
			plt.show()

In [ ]:
def check_e_transforms(v):
	locals = [[0,0],[1,0],[0,1],[1,1]]
	d_trap_order = {0:[0,1,2,3],1:[0,2,1,3],2:[0,1,2,3],3:[0,2,1,3]}
	d_tri_order = {0:[0,1,None,2],2:[0,1,2,None]}
	problems = {0:[],1:[]}
	inv_problems = {0:[],1:[]}
	for i,e in enumerate(v.mesh.zigzag_elements):
		for j,(xi,eta) in enumerate(locals):
			if e.tri:
				corner_id = d_tri_order[e.map_type][j]
			else:
				corner_id = d_trap_order[e.map_type][j]
			if corner_id is not None:
				corner = e.corners[corner_id]
				x,y = e.transform(xi,eta)
				if abs(x-corner.x) > 1e-12:
					problems[0][i] = corner.x
				if abs(y-corner.y) > 1e-12:
					problems[1][i] = corner.x

				e_xi,e_eta = e.inv_transform(corner.x,corner.y)
				if abs(e_xi-xi) > 1e-12:
					inv_problems[0][i] = corner.x
				if abs(e_eta-eta) > 1e-12:
					inv_problems[1][i] = corner.x
	try:
		assert len(problems[0]) == 0
		assert len(problems[1]) == 0
		assert len(inv_problems[0]) == 0
		assert len(inv_problems[1]) == 0
		return True,None
	except:
		return False,(problems,inv_problems)



In [ ]:
output = check_e_transforms(Zvar)
print(output)

In [ ]:
tri_done,trap_done = {i:False for i in range(4)},{i:False for i in range(4)}
for my_el in  Zvar.mesh.zigzag_elements:
	tridone,trapdone = tri_done[my_el.map_type],trap_done[my_el.map_type]
	if (my_el.tri and not tridone) or (not my_el.tri and not trapdone):
		if my_el.tri: tri_done[my_el.map_type]=True
		else: trap_done[my_el.map_type] = True
		fig = plt.figure(figsize=(10,1))
		for i,dof in enumerate(my_el.dof_list):
			plt.subplot(1,6,i+1)
			plt.plot(my_el.to_plot[0],my_el.to_plot[1])
			plt.plot(dof.x,dof.y,'ko')
			plt.axis('off')
		plt.show()

In [ ]:
dom = np.linspace(0,1)
xis,etas = [],[]
for xi in dom:
	for eta in dom:
		xis.append(xi)
		etas.append(eta)

tri_done,trap_done = {i:False for i in range(4)},{i:False for i in range(4)}
for my_el in  Zvar.mesh.zigzag_elements:
	tridone,trapdone = tri_done[my_el.map_type],trap_done[my_el.map_type]
	if (my_el.tri and not tridone) or (not my_el.tri and not trapdone):
		if my_el.tri: tri_done[my_el.map_type]=True
		else: trap_done[my_el.map_type] = True
		fig,ax = plt.subplots(1,2,figsize=(10,4))
		xs,ys = [],[]
		my_sum = []
		for (xi,eta) in zip(xis,etas):
			x,y = my_el.transform(xi,eta)
			xs.append(x)
			ys.append(y)
			my_sum.append(xi+eta)

		new_xis,new_etas = [],[]
		new_sum = []
		for (x,y) in zip(xs,ys):
			xi,eta = my_el.inv_transform(x,y)
			new_xis.append(xi)
			new_etas.append(eta)
			new_sum.append(x+y)

		ax[0].scatter(xs,ys,c=my_sum)	
		ax[0].plot(my_el.to_plot[0],my_el.to_plot[1],'k')

		ax[1].scatter(new_xis,new_etas,c=new_sum)
		ax[1].plot([0,1,1,0,0],[0,0,1,1,0],'k')

		plt.show()

#### integration checks

In [ ]:
tfunc = lambda x,y: x**2+y**2+x*y
t_func = lambda y,x: tfunc(x,y)
for p in Zvar.mesh.patches:
	for e_id in p.elements:
		e = p.elements[e_id]
		
		integrand = scint.dblquad(t_func,e.x,e.x+e.h,e.y,e.y+e.h)[0]

		func_vals = Zvar.integrator._evaluate_func_on_element(
							tfunc,e.bounds)
		my_integrand = 0
		vol = (e.h/2)**2
		for q_id in range(4):
			u_vals = func_vals[q_id]
			my_integrand += Zvar.integrator._compute_product_integral(
				u_vals,volume=vol)
		# print(round(integrand,8),round(my_integrand,8),sep='\t')
		if not (abs(integrand-my_integrand)<1e-15):
			print('uhoh')


In [ ]:
# let's check quadrature on each element
tfunc = lambda x,y: x**2+y**2+x*y
t_func = lambda y,x: tfunc(x,y)
for e in Zvar.mesh.zigzag_elements:
	xs = [c.x for c in e.corners]
	ys = [c.y for c in e.corners]
	xL,xR = min(xs),max(xs)
	yL,yR,yM = min(ys),max(ys),sum(ys)/3
	
	if e.tri:
		if xs.count(xL) > 1:
			y0func = lambda x: (yM-yL)/(xR-xL)*(x-xL)+yL
			y1func = lambda x: (yM-yR)/(xR-xL)*(x-xL)+yR
		else:
			y0func = lambda x: (yL-yM)/(xR-xL)*(x-xR)+yL
			y1func = lambda x: (yR-yM)/(xR-xL)*(x-xR)+yR
	else:
		if ys[0] == yL:
			y0func = lambda x: (x-xL)/2+yL
			y1func = lambda x: (xL-x)/2+yR
		else:
			y0func = lambda x: (xR-x)/2+yL
			y1func = lambda x: (x-xR)/2+yR

	integrand = scint.dblquad(t_func,xL,xR,y0func,y1func)[0]

	local_bounds = [0,1,0,1]
	func_vals = Zvar.integrator._evaluate_func_on_element(
						tfunc,local_bounds,wrap=e.transform)
	my_integrand = 0
	vol = .25
	for q_id in range(4):
		j_det = e.J_dets[q_id]
		u_vals = func_vals[q_id]
		my_integrand += Zvar.integrator._compute_product_integral(
			u_vals,volume=vol,jdet=j_det)
	if not (abs(integrand-my_integrand)<1e-15):
		print('uhoh')

#### stiffness integration checks

In [ ]:
from general_solve import shape_functions as sf

In [ ]:
N = 8
comp = 0
dofloc = 'xside' if comp==0 else 'yside'
ords = [2,1] if comp == 0 else [1,2]
rname = 'vertfinecenter' if comp==0 else 'horzfinecenter'
Zvar = Var(N,dofloc=dofloc,var=ufunc,ords=ords,rtype='stripe',rname=rname,zigzag=True)

In [ ]:
tfunc = lambda x,y: x**2+y**2+x*y
t_func = lambda y,x: tfunc(x,y)
for p in Zvar.mesh.patches:
	for e_id in p.elements:
		e = p.elements[e_id]
		for dof_i in e.dof_list:
			for dof_j in e.dof_list:
				tfunc = lambda x,y: dof_i.dphi([x,y])@dof_j.dphi([x,y])
				t_func = lambda y,x: tfunc(x,y)
		
				integrand = scint.dblquad(t_func,e.x,e.x+e.h,e.y,e.y+e.h)[0]

				func_vals = Zvar.integrator._evaluate_func_on_element(
							tfunc,e.bounds)
				my_integrand = 0
				vol = (e.h/2)**2
				for q_id in range(4):
					u_vals = func_vals[q_id]
					my_integrand += Zvar.integrator._compute_product_integral(
						u_vals,volume=vol)
				# print(round(integrand,8),round(my_integrand,8),sep='\t')
				if not (abs(integrand-my_integrand)<1e-15):
					print('uhoh')

In [ ]:
Zvar.solve_poisson(f_lap)

In [ ]:
id_map = Zvar.integrator.id_map
local_k = Zvar.integrator.k_vals
for p in Zvar.mesh.patches:
	for e_id in p.elements:
		e = p.elements[e_id]
		for i,dof_i in enumerate(e.dof_list):
			for j,dof_j in enumerate(e.dof_list):
				tfunc = lambda x,y: dof_i.dphi([x,y])@dof_j.dphi([x,y])
				t_func = lambda y,x: tfunc(x,y)
		
				integrand = scint.dblquad(t_func,e.x,e.x+e.h,e.y,e.y+e.h)[0]
				my_integrand = 0
				for q_id in range(4):
					my_integrand += local_k[q_id][i,j]
				if not (abs(integrand-my_integrand)<1e-15):
					print('uhoh')

In [ ]:
# let's check quadrature on each element
tfunc = lambda x,y: x**2+y**2+x*y
t_func = lambda y,x: tfunc(x,y)
for e in Zvar.mesh.zigzag_elements:
	if e.tri:
		xs = [c.x for c in e.corners]
		ys = [c.y for c in e.corners]
		xL,xR = min(xs),max(xs)
		yL,yR,yM = min(ys),max(ys),sum(ys)/3
		
		if e.tri:
			if xs.count(xL) > 1:
				y0func = lambda x: (yM-yL)/(xR-xL)*(x-xL)+yL
				y1func = lambda x: (yM-yR)/(xR-xL)*(x-xL)+yR
			else:
				y0func = lambda x: (yL-yM)/(xR-xL)*(x-xR)+yL
				y1func = lambda x: (yR-yM)/(xR-xL)*(x-xR)+yR
		else:
			if ys[0] == yL:
				y0func = lambda x: (x-xL)/2+yL
				y1func = lambda x: (xL-x)/2+yR
			else:
				y0func = lambda x: (xR-x)/2+yL
				y1func = lambda x: (x-xR)/2+yR

		for i,dof_i in enumerate(e.dof_list):
			ixi0,ieta0 = dof_i.ref_shifts[e.global_ID]
			if e.tri:
				dphi_i = lambda x,y: sf.dphi_tri(x,y,e.h,e.bary_coefs,
												dof_i.x,dof_i.y,e.map_type)
			else:
				dphi_i = lambda x,y: sf.dphi_trap(x,y,e.h,e.x,e.y,
												ixi0,ieta0,e.map_type)
			for j,dof_j in enumerate(e.dof_list):
				jxi0,jeta0 = dof_j.ref_shifts[e.global_ID]
				if e.tri:
					dphi_j = lambda x,y: sf.dphi_tri(x,y,e.h,e.bary_coefs,
													dof_j.x,dof_j.y,e.map_type)
					dphi_all = lambda x,y: sf.dphi_tri(x,y,e.h,e.bary_coefs,
											dof_i.x,dof_i.y,e.map_type,dof_j.x,dof_j.y)
				else:
					dphi_j = lambda x,y: sf.dphi_trap(x,y,e.h,e.x,e.y,
													jxi0,jeta0,e.map_type)

					dphi_all = lambda x,y: sf.dphi_trap(x,y,e.h,e.x,e.y,
											ixi0,ieta0,e.map_type,jxi0,jeta0)
				tfunc_i0 = lambda x,y: dphi_i(x,y)[0]#@dphi_j(x,y)
				tfunc_i1 = lambda x,y: dphi_i(x,y)[1]#@dphi_j(x,y)
				tfunc_j0 = lambda x,y: dphi_j(x,y)[0]#@dphi_j(x,y)
				tfunc_j1 = lambda x,y: dphi_j(x,y)[1]#@dphi_j(x,y)
				tfunc = tfunc_i0#lambda x,y: dphi_i(x,y)@dphi_j(x,y)
				# tfuncs = [tfunc_i0,tfunc_i1,tfunc_j0,tfunc_j1]

				for iter in range(1):
					# tfunc = lambda x,y: tfunc_i0(x,y)*tfunc_j0(x,y)+tfunc_i1(x,y)*tfunc_j1(x,y)
					# tfunc = tfuncs[iter]
					_tfunc = dphi_all#_tfuncs[iter]

					t_func = lambda y,x: tfunc(x,y)
					integrand,err = scint.dblquad(t_func,xL,xR,y0func,y1func,epsabs=1e-14)

					local_bounds = [0,1,0,1]
					func_vals = Zvar.integrator._evaluate_func_on_element(
										tfunc,local_bounds,wrap=e.transform)
					# _func_vals = Zvar.integrator._evaluate_func_on_element(
										# _tfunc,local_bounds,wrap=e.transform)
					my_integrand = 0
					# _my_integrand = 0
					vol = .25
					for q_id in range(4):
						j_det = e.J_dets[q_id]
						u_vals = func_vals[q_id]
						# _u_vals = _func_vals[q_id]
						my_integrand += Zvar.integrator._compute_product_integral(
							u_vals,volume=vol,jdet=j_det)
						# _my_integrand += Zvar.integrator._compute_product_integral(
							# _u_vals,volume=vol,jdet=j_det)
					if not (abs(integrand-my_integrand)<1e-15):
						print('uhoh')
						print(abs(integrand-my_integrand))

In [ ]:
# let's check quadrature on each element
k_vals = Zvar.interface_map.k_vals
for e in Zvar.mesh.zigzag_elements:
	if not e.tri:
		xs = [c.x for c in e.corners]
		ys = [c.y for c in e.corners]
		xL,xR = min(xs),max(xs)
		yL,yR,yM = min(ys),max(ys),sum(ys)/3
		
		# if e.tri:
		# 	if xs.count(xL) > 1:
		# 		y0func = lambda x: (yM-yL)/(xR-xL)*(x-xL)+yL
		# 		y1func = lambda x: (yM-yR)/(xR-xL)*(x-xL)+yR
		# 	else:
		# 		y0func = lambda x: (yL-yM)/(xR-xL)*(x-xR)+yL
		# 		y1func = lambda x: (yR-yM)/(xR-xL)*(x-xR)+yR
		# else:
		# 	if ys[0] == yL:
		# 		y0func = lambda x: (x-xL)/2+yL
		# 		y1func = lambda x: (xL-x)/2+yR
		# 	else:
		# 		y0func = lambda x: (xR-x)/2+yL
		# 		y1func = lambda x: (x-xR)/2+yR

		for i,dof_i in enumerate(e.dof_list):
			ixi0,ieta0 = dof_i.ref_shifts[e.global_ID]
			dphi_i = lambda x,y: sf.dphi_trap(x,y,e.h,e.x,e.y,
									 		  ixi0,ieta0,e.map_type)
			for j,dof_j in enumerate(e.dof_list):
				jxi0,jeta0 = dof_j.ref_shifts[e.global_ID]
				# dphi_j = lambda x,y: sf.dphi_trap(x,y,e.h,e.x,e.y,
				# 					  			  jxi0,jeta0,e.map_type)

				tfunc = lambda x,y: sf.dphi_trap(x,y,e.h,e.x,e.y,
										ixi0,ieta0,e.map_type,jxi0,jeta0)
				# tfunc_i0 = lambda x,y: dphi_i(x,y)[0]#@dphi_j(x,y)
				# tfunc_i1 = lambda x,y: dphi_i(x,y)[1]#@dphi_j(x,y)
				# tfunc_j0 = lambda x,y: dphi_j(x,y)[0]#@dphi_j(x,y)
				# tfunc_j1 = lambda x,y: dphi_j(x,y)[1]#@dphi_j(x,y)

				# for tfunc in [tfunc_i0,tfunc_i1,tfunc_j0,tfunc_j1]:

				# t_func = lambda y,x: tfunc(x,y)
				# integrand,err = scint.dblquad(t_func,xL,xR,y0func,y1func,epsabs=1e-14)
				integrand = 0

				local_bounds = [0,1,0,1]
				func_vals = Zvar.integrator._evaluate_func_on_element(
									tfunc,local_bounds,wrap=e.transform)
				my_integrand = 0
				vol = .25
				for q_id in range(4):
					integrand += k_vals[False][e.map_type][q_id][i,j]
					j_det = e.J_dets[q_id]
					u_vals = func_vals[q_id]
					my_integrand += Zvar.integrator._compute_product_integral(
						u_vals,volume=vol,jdet=j_det)
				if not (abs(integrand-my_integrand)<1e-15):
					print('uhoh')
					print(abs(integrand-my_integrand))

In [ ]:
e = Zvar.mesh.zigzag_elements[-15]
print(e.map_type)

dom = np.linspace(0,1)[1:-1]
xis, etas = np.meshgrid(dom,dom)

xs,ys = [],[]
for xi in dom:
	etas = np.linspace(0,1-xi)
	for eta in etas:
		x,y = e.transform(xi,eta)
		xs.append(x)
		ys.append(y)

print([e.h*v for v in e.bary_coefs[2:-1]])
plt.plot(e.to_plot[0],e.to_plot[1],'k')
for c in e.dof_list:#corners:
	xi,eta = e.inv_transform(c.x,c.y)
	print(xi,eta,c.ref_shifts[e.global_ID])
	plt.plot(c.x,c.y,'o')

plt.plot(xs,ys,'o')
plt.show()

In [ ]:
from matplotlib import cm
e = Zvar.mesh.zigzag_elements[-5]
h = e.h
dom = np.linspace(0,1)[1:-1]
xis,etas = [],[]
xs,ys = [],[]
for xi in dom:
	tmpetas = np.linspace(0,1-xi)
	for eta in tmpetas:
		xis.append(xi)
		etas.append(eta)
		x,y = e.transform(xi,eta)
		xs.append(x)
		ys.append(y)
n = len(dom)
m = len(tmpetas)

X,Y = np.array(xs).reshape(n,m), np.array(ys).reshape(n,m)

vals = {i:[] for i in range(4)}
xp,yp,coef0,coef1,coef2,coef3,tmp = e.bary_coefs
for x,y in zip(xs,ys):
	px, py = x-xp, y-yp

	A = coef0*px + coef1*py
	B = coef2*px + coef3*py
	C = 1-A-B
	vals[0].append(A)
	vals[1].append(B)
	vals[2].append(C)
	vals[3].append(.75-y)
	# vals.append(.5+(y-e.mid[1]+.5*(x-e.min[0]))/e.h)
for i in range(4):
	vals[i] = np.array(vals[i]).reshape(n,m)
fig, ax = plt.subplots(1,3,figsize=(30,10),subplot_kw={"projection": "3d"})
# surf = ax.plot_surface(X, Y, vals, cmap=cm.coolwarm,
# 				linewidth=0, antialiased=False)
# fig.colorbar(surf)
for i in range(3):
	surf = ax[i].plot_surface(X, Y, vals[i], cmap=cm.coolwarm,
					linewidth=0, antialiased=False)
	fig.colorbar(surf,location='bottom')
	ax[i].scatter(e.x,e.y,1,marker='o',color='k')#,markersize=10)
plt.show()

plt.plot(e.to_plot[0],e.to_plot[1])
plt.plot(xp,yp,'o')

In [ ]:
e.regular

In [ ]:
sf.bspline3(0,1)

In [ ]:

Zvar = Var(N,dofloc=dofloc,var=ufunc,ords=ords,rtype='stripe',rname=rname,zigzag=True)

In [ ]:
from matplotlib import cm
e = Zvar.mesh.zigzag_elements[-5]
h = e.h
dom = np.linspace(0,1,111)[1:-1]
xis,etas = [],[]
xs,ys = [],[]
for xi in dom:
	tmpetas = np.linspace(0,1-xi,111)[1:-1]
	for eta in tmpetas:
		xis.append(xi)
		etas.append(eta)
		x,y = e.transform(xi,eta)
		xs.append(x)
		ys.append(y)
n = len(dom)
m = len(tmpetas)


X,Y = np.array(xs).reshape(n,m), np.array(ys).reshape(n,m)
locs,titles = ['left','bottom','right'],['deta/dx','deta/dy','phi']

check = 0.
center = 0.
for dof in e.dof_list:
	print(dof.ref_shifts[e.global_ID])
	print(e.inv_transform(dof.x,dof.y))
	x0,y0 = dof.x,dof.y
	fig, ax = plt.subplots(1,3,figsize=(30,10),subplot_kw={"projection": "3d"})
	vals = {0:[],1:[],2:[]}
	for (x,y) in zip(xs,ys):
		dphi = sf.dphi_tri(x,y,e.h,e.bary_coefs,x0,y0,e.map_type,check=check)
		phival = dof.phi([x,y],glob=True,el=e)
		vals[0].append(dphi[0])
		vals[1].append(dphi[1])
		# vals[0].append(phival[0])
		# vals[1].append(phival[1])
		vals[2].append(phival)#[0]*phival[1])

	for c in e.corners:
		print((c.x,c.y),dof.phi([c.x,c.y],glob=True,el=e),sep='\t')
	for j in range(3):
		myv = np.array(vals[j]).reshape(n,m)
		if j == 2: 
			if sum(dof.ref_shifts[e.global_ID]) != 0:
				check += myv
			else:
				center += myv
		surf = ax[j].plot_surface(X, Y, myv, cmap=cm.coolwarm,
						linewidth=0, antialiased=False)
		ax[j].set_title(titles[j])
		fig.colorbar(surf,location=locs[j])
	plt.suptitle([x0,y0])
	plt.show()
fig,ax = plt.subplots(1,3,figsize=(30,10),subplot_kw={"projection":"3d"})
surf = ax[0].plot_surface(X, Y, 1-check, cmap=cm.coolwarm,
				linewidth=0, antialiased=False)
fig.colorbar(surf,location="left")
surf = ax[1].plot_surface(X, Y, center, cmap=cm.coolwarm,
				linewidth=0, antialiased=False)
fig.colorbar(surf,location="bottom")
surf = ax[2].plot_surface(X, Y, check+center, cmap=cm.coolwarm,
				linewidth=0, antialiased=False)
fig.colorbar(surf,location="right")
plt.show()

In [ ]:
xs[0],ys[0]

In [ ]:
e.map_type

In [ ]:
plt.plot(e.to_plot[0],e.to_plot[1])
for dof in e.dof_list:
	ixi0,ieta0 = dof.ref_shifts[e.global_ID]
	plt.plot(dof.x,dof.y,'o',label=[ixi0,ieta0])
plt.legend()
plt.show()

In [ ]:
from matplotlib import cm
from matplotlib.ticker import LinearLocator


# Make data.
dom = np.linspace(0,1)[1:-1]
xis, etas = np.meshgrid(dom,dom)

n,m = xis.shape
xis,etas = xis.flatten(),etas.flatten()

e = Zvar.mesh.zigzag_elements[-3]
xs,ys = [],[]
for (xi,eta) in zip(xis,etas):
	x,y = e.transform(xi,eta)
	xs.append(x)
	ys.append(y)

X,Y = np.array(xs).reshape(n,m), np.array(ys).reshape(n,m)

for dof in e.dof_list:
	ixi0,ieta0 = dof.ref_shifts[e.global_ID]
	print(ixi0,ieta0)
	dphi = lambda x,y: sf.dphi_tri(x,y,e.h,e.min[0],e.mid[1],#e.x,e.y,
										ixi0,ieta0,e.map_type,inv_func=e.inv_transform)
	if True:#for dof2 in e.dof_list:
		# jxi0,jeta0 = dof2.ref_shifts[e.global_ID]
		# dphi_all = lambda x,y: sf.dphi_tri(x,y,e.h,e.min[0],e.min[1],#e.x,e.y,
										# ixi0,ieta0,e.map_type,jxi0,jeta0)
		# vals = np.array([dphi_all(x,y) for (x,y) in zip(xs,ys)]).reshape(n,m)
		vals = np.array([dphi(x,y)[0] for (x,y) in zip(xs,ys)]).reshape(n,m)
		fig, ax = plt.subplots(figsize=(20,20),subplot_kw={"projection": "3d"})
		surf = ax.plot_surface(X, Y, vals, cmap=cm.coolwarm,
							linewidth=0, antialiased=False)
		fig.colorbar(surf,location='left')
		plt.show()
# 	# phi0 = lambda x,y: sf.phi_trap(x,y,e.h,e.x,e.y,
# 	# 									ixi0,ieta0,e.map_type)
# 	# phi1 = lambda x,y: dof.phi([x,y],glob=True)
# 	# vals0 = np.array([phi0(x,y) for (x,y) in zip(xs,ys)]).reshape(n,m)
# 	# vals1 = np.array([phi1(x,y) for (x,y) in zip(xs,ys)]).reshape(n,m)
# 	# diffs = np.array([abs(phi0(x,y)-phi1(x,y)) for (x,y) in zip(xs,ys)]).reshape(n,m)
# 	vals = [dphi(x,y) for (x,y) in zip(xs,ys)]
# 	dx = np.array([v[0] for v in vals]).reshape(n,m)
# 	dy = np.array([v[1] for v in vals]).reshape(n,m)

# 	# fig, ax = plt.subplots(figsize=(20,20),subplot_kw={"projection": "3d"})
# 	fig, ax = plt.subplots(1,2,figsize=(20,10),subplot_kw={"projection": "3d"})
# 	# surf = ax.plot_surface(X, Y, vals, cmap=cm.coolwarm,
# 	# 					linewidth=0, antialiased=False)
	
# # 	fig.colorbar(surf,location='left')
		
# 	surf0 = ax[0].plot_surface(X, Y, dx, cmap=cm.coolwarm,
# 						linewidth=0, antialiased=False)
	
# 	fig.colorbar(surf0,location='left')
# 	surf1 = ax[1].plot_surface(X, Y, dy, cmap=cm.coolwarm,
# 						linewidth=0, antialiased=False)
# 	fig.colorbar(surf1,location='right')

# # # # Customize the z axis.
# # # ax.set_zlim(-1.01, 1.01)
# # # ax.zaxis.set_major_locator(LinearLocator(10))
# # # # A StrMethodFormatter is used automatically
# # # ax.zaxis.set_major_formatter('{x:.02f}')

# # # # Add a color bar which maps values to colors.
# # # fig.colorbar(surf, shrink=0.5, aspect=5)

# 	plt.show()

In [ ]:
dof.ref_shifts

In [ ]:
ind_to_shifts = {0:[1,0],1:[0,0],2:[-1,0],3:[1,-1],4:[0,-1],5:[-1,-1]}
e = Zvar.mesh.zigzag_elements[3]
plt.plot(e.to_plot[0],e.to_plot[1])
for loc_id,dof in enumerate(e.dof_list):
	plt.plot(dof.x,dof.y,'o',label=dof.ref_shifts[e.global_ID])
	print((dof.x-e.x)/e.h,dof.ref_shifts[e.global_ID],ind_to_shifts[loc_id])
plt.legend()
plt.show()